# Many-Body Physics & MPS Simulation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/05_many_body_mps.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/05_many_body_mps.ipynb)


This notebook covers:
1. **MPS: GHZ state** — exact bond dimension 2
2. **Kibble-Zurek mechanism** — defect density power law in 1D TFIM
3. **Finite-size scaling** — Binder cumulant at the Ising phase transition


In [ ]:
# Install quantx-engine (needed in Colab / Binder)
# Skipped automatically if already installed in the local environment.
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

In [ ]:
import numpy as np

## 1. MPS: GHZ State

The $n$-qubit GHZ state $|\text{GHZ}\rangle = (|0\rangle^{\otimes n} + |1\rangle^{\otimes n})/\sqrt{2}$
has exact MPS bond dimension 2.  `ghz_fidelity(n, max_bond=2)` should return 1.0.

In [ ]:
from quantx_engine.mps_simulator import ghz_fidelity

for n in [4, 6, 8, 10, 12]:
    f = ghz_fidelity(n, max_bond=2)
    print(f'GHZ fidelity (n={n:2d}, bond=2): {f:.8f}')
    assert abs(f - 1.0) < 1e-8, f'Expected fidelity=1.0, got {f}'
print('All GHZ fidelities = 1.0 confirmed.')

## 2. Kibble-Zurek Mechanism

When ramping through a quantum phase transition at rate $v$, the density of topological
defects follows a power law:
$$n_{\text{def}} \propto v^{\frac{\nu}{1+\nu z}}$$
For the 1D TFIM with $\nu = z = 1$ the predicted exponent is $1/2$.

We use `KibbleZurekScaling` with a small system (n=8) to keep it fast.

In [ ]:
from quantx_engine.neqdyn import KibbleZurekScaling

# n_sites must stay small (exact diagonalisation)
kz = KibbleZurekScaling(n_sites=8, J=1.0, n_steps=25)
quench_rates = [0.05, 0.10, 0.20, 0.50, 1.00, 2.00]
result = kz.sweep(quench_rates=quench_rates)

print('Quench rate v | Defect density')
print('-' * 30)
for v, d in zip(result.quench_rates, result.defect_densities):
    print(f'  v={v:.2f}       |  {d:.5f}')

print(f'\nFitted KZ exponent: {result.kz_exponent:.3f}  (theory prediction: 0.5 for 1D TFIM)')

In [ ]:
# Verify power-law scaling: fit log-log slope
# Filter to increasing portion of the sweep (large v where power law holds)
rates = np.array(result.quench_rates)
dens  = np.array(result.defect_densities)
# Use the upper half of quench rates where defect density increases monotonically
mask = rates >= 0.2
if mask.sum() >= 3:
    log_v = np.log(rates[mask])
    log_d = np.log(np.maximum(dens[mask], 1e-10))
    exponent = np.polyfit(log_v, log_d, 1)[0]
    print(f'Power-law exponent (v >= 0.2): {exponent:.3f}  (theory: ~0.5)')
    # Broad tolerance since n=8 has finite-size corrections
    assert 0.2 < exponent < 1.5, f'KZ exponent {exponent:.3f} out of expected range'
    print('Kibble-Zurek power-law assertion passed.')
else:
    print('Not enough data points for reliable fit — skipping assertion.')

## 3. Finite-Size Scaling: Binder Cumulant

The Binder cumulant $U_4 = 1 - \langle m^4 \rangle / (3 \langle m^2 \rangle^2)$
approaches $2/3$ in the ordered phase and $0$ in the disordered phase.

In [ ]:
from quantx_engine.finite_size import binder_cumulant

np.random.seed(42)

# Ordered phase: magnetisation tightly clustered around +/-M0
M0 = 0.9
samples_ordered = np.concatenate([
    np.random.normal(+M0, 0.05, 500),
    np.random.normal(-M0, 0.05, 500),
])
U_ordered = binder_cumulant(samples_ordered)
print(f'Binder cumulant (ordered phase): {U_ordered:.4f}  (theory: 2/3 = {2/3:.4f})')

# Disordered phase: magnetisation near zero
samples_disordered = np.random.normal(0.0, 0.3, 1000)
U_disordered = binder_cumulant(samples_disordered)
print(f'Binder cumulant (disordered phase): {U_disordered:.4f}  (theory: 0)')

assert U_ordered > U_disordered + 0.3, 'Ordered cumulant should be much larger than disordered'
assert abs(U_ordered - 2/3) < 0.1,    'Ordered cumulant should approach 2/3'
print('Binder cumulant assertions passed.')

## Summary

| Module | Demonstrated |
|--------|--------------|
| `mps_simulator.ghz_fidelity` | Exact GHZ fidelity = 1.0 at bond dimension 2 (n up to 12) |
| `neqdyn.KibbleZurekScaling` | Power-law defect density with Kibble-Zurek exponent |
| `finite_size.binder_cumulant` | Ordered → 2/3, disordered → 0 |

This completes the five tutorial notebooks! Explore the full [API Reference](../docs/source/api/index.rst).